# FuelAnalyst tutorial: from Survey123 plots to fire behaviour and effects

*Adapted from a tutorial by Sarah Smith-Tripp (UBC Centre for Wildfire Coexistence), generalized and updated for the
current FuelAnalyst release. Released under the same MIT licence as FuelAnalyst.*

## Overview

FuelAnalyst generates fuel attributes from plot data collected with the fuel sampling protocols and Survey123 data
collection forms in this repository (`docs/survey123/`). Its outputs support:

1. tree- and stand-level summaries of fuels;
2. CFFBPS fuel type assignment with the British Columbia fuel typing algorithm (Perrakis et al. 2018);
3. fire behaviour modelling with the Canadian Conifer Pyrometrics (ConPyro; Perrakis et al. 2019, 2023) and Crown Fire
   Initiation and Spread (CFIS; Cruz et al. 2003, 2004, 2005; Alexander et al. 2006) systems; and
4. fire effects and greenhouse gas emissions modelling with FOFEM (Lutes et al. 2020).

Fuel biomass is calculated with standard line intercept equations for downed woody debris (Van Wagner 1968, 1982), the
Photoload method for surface vegetation (Keane and Dickinson 2007), the Canadian National Biomass equations for tree
bole wood, branches, foliage, and bark (Lambert et al. 2005; Ung et al. 2008), and USDA Forest Service FIA equations for
duff and litter bulk density (Woodall and Monleon 2008).

FuelAnalyst works with companion packages that you install alongside it: **ConPyro** (crown fire occurrence and
fire behaviour), **flame-components** (flame geometry and related fire behaviour functions), **pyfofem** (fire effects
and emissions), and the BC fuel typing tool (BCWFT2018, bundled with FuelAnalyst).

## What this notebook covers

| Step | Section | What you do |
|---|---|---|
| 1 | Setup | Load packages and locate the example data |
| 2 | Explore the data | Look at the Survey123 folder structure and link trees to plots |
| 3 | Run FuelAnalyst | Create tree, stand, and fuel attributes with `fuel_analyst()` |
| 4 | FBP fuel typing | Assign fuel types with `process_fbp_fuel_type()` |
| 5 | Weather | Understand weather extraction, then merge weather with plots using `merge_weather_data()` |
| 6 | ConPyro and FOFEM | Model fire behaviour, tree mortality, and emissions with `run_conpyro()` |

Run the cells in order. Answers to the short questions are hidden under **Answer** headings; click one to reveal it.
Everything runs on the example dataset that ships with the repository, and the whole notebook takes a few minutes.

## 1. Setup

The next cell imports the packages used below. If it raises an `ImportError`, FuelAnalyst or one of its companion
packages is not installed in the Python environment this notebook is using. Follow [SETUP.md](SETUP.md) (installing
Python, creating an environment, and selecting it in VS Code or PyCharm), then restart the kernel.

In [ ]:
import copy
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from fuel_analyst.fuel_analyst import fuel_analyst
    from fuel_analyst.components import fofem_units, helper
    from fuel_analyst.run_conpyro import default_fire_modelling_dict, run_conpyro
    from fuel_analyst.tools import merge_wx_data
    from fuel_analyst.tools.fbp_fuel_type import process_fbp_fuel_type
    import conpyro, flame_components, pyfofem  # noqa: F401  (companion packages)
except ImportError as err:
    raise ImportError(
        'FuelAnalyst or a companion package (conpyro, flame-components, pyfofem) is not installed in this '
        'environment. See examples/tutorial/SETUP.md, then restart the kernel.'
    ) from err

print('FuelAnalyst and companion packages loaded.')

### Locating the example data

The example dataset lives in `examples/data/` at the top of the repository. The helper below searches upward from the
folder you are running in, so the notebook works wherever you open it from inside the repository. Results are written
to `examples/results/tutorial/`, which Git ignores.

In [ ]:
def find_example_data(start=None):
    """Return the ``examples/data`` folder by searching upward from ``start`` (default: the working directory)."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        for candidate in (folder / 'data', folder / 'examples' / 'data'):
            if (candidate / 'Survey123_Data').is_dir():
                return candidate
    raise FileNotFoundError(
        "Could not find 'examples/data/Survey123_Data'. Open this notebook from inside a clone of the "
        'FuelAnalyst repository (the data folder is part of the repository, not the installed package).'
    )


DATA_DIR = find_example_data()
RAW_DIR = DATA_DIR / 'Survey123_Data'
OUT_DIR = DATA_DIR.parent / 'results' / 'tutorial'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Example data: {DATA_DIR}')
print(f'Results will be written to: {OUT_DIR}')

## 2. Explore the example data

FuelAnalyst expects Survey123 exports in a fixed folder structure: five sub-folders, one per survey form. Here is the
example dataset, which contains 20 plots.

In [ ]:
def print_tree(directory, prefix=''):
    """Print a folder and its contents as an indented tree."""
    entries = sorted(Path(directory).iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    for i, entry in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ('└── ' if last else '├── ') + entry.name)
        if entry.is_dir():
            print_tree(entry, prefix + ('    ' if last else '│   '))


print(RAW_DIR.name)
print_tree(RAW_DIR)

The five sub-folders, one per Survey123 form, are:

In [ ]:
DIR_DICT = {
    'plot': '1_Plot_Attributes',
    'plants': '2_Plants_Regeneration',
    'trees': '3_Trees',
    'cover': '4_Canopy_Cover',
    'woody': '5_Woody_Debris',
}
DIR_DICT

Each sub-folder has a `survey_0.csv` with one row per plot for that form (trees, canopy cover, woody debris, and so
on). Each plot is identified by its `GlobalID`. Sub-tables such as `3_Trees/tree_plot_2.csv` or
`2_Plants_Regeneration/regeneration_1.csv` also have a `GlobalID`, but it identifies the individual record (one tree,
for example), not the plot. The `ParentGlobalID` column links each record back to its plot's `GlobalID`.

### Plot-level information

How many plots are in the example data?

In [ ]:
plot_data = pd.read_csv(RAW_DIR / DIR_DICT['plot'] / 'survey_0.csv')
plot_data.describe(include='all').head(2)

<details>
<summary><b>Answer: total plots</b></summary>

There are **20** plots (the `count` row).

</details>

### Linking trees to plots

Because each form is exported separately, FuelAnalyst builds its own plot key, `unique_id`, from the survey date,
treatment area, treatment unit, treatment type, and plot ID. `helper.get_unique_id()` calculates it. Below we add
`unique_id` to the trees form, read the individual tree records, and join each tree to its plot with a `merge`.

In [ ]:
tree_plots = helper.get_unique_id(pd.read_csv(RAW_DIR / DIR_DICT['trees'] / 'survey_0.csv'))
trees = pd.read_csv(RAW_DIR / DIR_DICT['trees'] / 'tree_plot_2.csv')

trees_by_plot = trees.merge(
    tree_plots,
    left_on='ParentGlobalID',   # links each tree to its plot
    right_on='GlobalID',        # the plot's own ID
    how='left',
    validate='m:1',             # raises MergeError if a plot's GlobalID is not unique
    indicator=True,             # adds '_merge': 'both' or 'left_only'
    suffixes=('_tree', '_plot'),
)
print('Trees that did not match a plot:', int((trees_by_plot['_merge'] == 'left_only').sum()))
print('Trees in total:', len(trees_by_plot))

With the trees aligned to their plots, we can summarize them. Which plot has the most trees?

In [ ]:
tree_summary = (
    trees_by_plot.groupby('unique_id')
    .agg(total_count=('unique_id', 'count'), avg_dbh=('tree_dbh', 'mean'))
    .reset_index()
)
tree_summary

<details>
<summary><b>Answer: plot with the most trees</b></summary>

`Demo_TU6_burned_P16_2021`, with **54** trees. Sort with `tree_summary.sort_values("total_count", ascending=False)` to check.

</details>

Try a few questions of your own. `groupby()` and `agg()` are the pandas equivalents of dplyr's `group_by()` and
`summarise()` (see the [pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html)).

## 3. Run FuelAnalyst

`fuel_analyst()` reads the five Survey123 folders and calculates tree, stand, woody debris, and surface fuel
attributes. Its main inputs are:

- `in_dir`: the folder holding the five Survey123 sub-folders;
- `out_dir`: where to write results;
- `surface_fire_intensity_model`: the equation used to estimate surface fire intensity; and
- optional settings for stand summaries, woody debris, ConPyro, and FOFEM inputs. We use the defaults here. See
  `docs/CODEBASE.md` for what each setting controls.

The `gen_*` switches choose which derived tables to write. Plot Attributes is always processed, and the tables you
request need their source forms. The ConPyro and FOFEM consumption tables used below need all five.

In [ ]:
FA_DIR = OUT_DIR / 'fuel_analyst'

fuel_analyst(
    in_dir=RAW_DIR,
    out_dir=FA_DIR,
    surface_fire_intensity_model='Byram',
    gen_conpyro_df=True,             # inputs for ConPyro fire behaviour modelling
    gen_fofem_gui_app_tree_df=False, # tree table formatted for the external FOFEM desktop app (not needed here)
    gen_fofem_consume_df=True,       # surface and ground fuel inputs for FOFEM consumption and emissions
)

### The error report

`fuelanalyst_errors.csv` is the most important output. It lists data problems found while processing, one row per
affected plot and one column per part of the pipeline that reported it. Fix these before moving on.

In [ ]:
errors = pd.read_csv(FA_DIR / 'fuelanalyst_errors.csv')
print(f'{len(errors)} plots reported problems.')
errors.iloc[:, [0, -1]].head(10)

<details>
<summary><b>Answer: are there errors to address?</b></summary>

Yes. **8 plots** report `missing tree dbh`: a tree record has no diameter, so FuelAnalyst assigns it zero density and logs the `tree_id` (the field crew's tag number) to help you find it. In your own data, go back to the field form, correct or confirm the record, and re-run. The example dataset keeps these records so you can see what a real report looks like; the rest of the workflow still runs.

</details>

### The outputs

Running `fuel_analyst()` wrote these files to the output folder:

In [ ]:
for path in sorted(FA_DIR.glob('*.csv')):
    print(path.name)

- **`master_fuel_df.csv`**: stand-level fuel and forest structure attributes, plus fuel biomass and density for both
  Canadian and US fuel classifications. The Canadian attributes feed Conifer Pyrometrics; the US attributes feed FOFEM.
  It also has flame length and fire intensity estimates and the inputs needed to assign CFFBPS fuel types.

In [ ]:
master_fuel_df = pd.read_csv(FA_DIR / 'master_fuel_df.csv')
master_fuel_df.dropna(axis=1).head(2)

- **`master_tree_df.csv`**: tree-level attributes (species, status, decay class, diameter, density, basal area,
  bark thickness, height). It keeps live and dead trees, with a `status` column to tell them apart. It is the
  tree input for FOFEM mortality modelling.

In [ ]:
master_tree_df = pd.read_csv(FA_DIR / 'master_tree_df.csv')
master_tree_df.dropna(axis=1).head(2)

- **`conpyro_df.csv`**: one row per plot with the stand inputs for fire behaviour modelling. It has no weather yet,
  and of course no model outputs.

In [ ]:
conpyro_df = pd.read_csv(FA_DIR / 'conpyro_df.csv')
conpyro_df.dropna(axis=1).head(2)

- **`fbp_prep_df.csv`**: the inputs for FBP fuel typing. Some fields are empty on purpose because FuelAnalyst cannot
  know them from the field data. You must supply them from outside sources (next section).

In [ ]:
fbp_prep_df = pd.read_csv(FA_DIR / 'fbp_prep_df.csv')
fbp_prep_df.head(3)

- **`fofem_consume_si_df.csv`**: surface and ground fuel inputs for FOFEM consumption and emissions. The unit system
  is part of the file name (`si` or `imperial`); the default is SI.

In [ ]:
fofem_consume_df = pd.read_csv(FA_DIR / fofem_units.consume_filename('SI'))
fofem_consume_df.dropna(axis=1).head(2)

## 4. FBP fuel typing

The next step assigns a CFFBPS fuel type to each plot from its stand attributes. FuelAnalyst does this with the BC
fuel typing algorithm (BCWFT2018), which also needs information that does not come from the field survey. In
`fbp_prep_df.csv`, these fields are left empty for you to fill in:

| Field | What it is |
|---|---|
| `COAST_INTERIOR_CD` | `C` for coastal or `I` for interior |
| `BEC_Zone`, `BEC_Subzone` | Biogeoclimatic zone and subzone |
| `SPECIES_CD_1_max_age` | Age of the oldest leading-species tree (years) |
| `recent_clearcut_date_10_yrs` | Date of a clearcut in the last 10 years, if any |
| `recent_fire_date_11_yrs` | Date of a fire in the last 11 years, if any |

These normally come from provincial GIS layers (for example VRI and BEC zone data) at each plot's location. **Check
the plot locations first**: a wrong coordinate means wrong BEC and disturbance values. The scatter plot below is a quick
check; in your own work, overlay the points on the source layers.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
treatment = conpyro_df['unique_id'].str.extract(r'_(burned|treated|untreated)_')[0]
for name, colour in {'burned': '#c0392b', 'treated': '#148f77', 'untreated': '#2471a3'}.items():
    mask = treatment == name
    ax.scatter(conpyro_df.loc[mask, 'long'], conpyro_df.loc[mask, 'lat'], label=name, color=colour, s=40)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Plot locations')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

The example dataset ships a completed version of this table, with the fields above already filled in
(`examples/data/fbp_fuel_typing/fbp_prep_df.csv`). We copy it next to the FuelAnalyst outputs and run the fuel typing.
`season` is `'growing'` or `'dormant'`.

In [ ]:
prep_filled = FA_DIR / 'fbp_prep_df_BEC.csv'
shutil.copy(DATA_DIR / 'fbp_fuel_typing' / 'fbp_prep_df.csv', prep_filled)
pd.read_csv(prep_filled)[['unique_id', 'COAST_INTERIOR_CD', 'BEC_Zone', 'BEC_Subzone', 'SPECIES_CD_1_max_age']].head(5)

In [ ]:
FBP_DIR = OUT_DIR / 'fbp_fuel_typing'

process_fbp_fuel_type(
    master_fuel_df_path=str(FA_DIR / 'master_fuel_df.csv'),
    fbp_prep_df_path=str(prep_filled),
    conpyro_df_path=str(FA_DIR / 'conpyro_df.csv'),
    out_dir=str(FBP_DIR),
    season='growing',
)

The fuel types are written into a copy of the ConPyro table, `conpyro_df_fbp.csv`:

In [ ]:
conpyro_df_fbp = pd.read_csv(FBP_DIR / 'conpyro_df_fbp.csv')
conpyro_df_fbp[['unique_id', 'fbp_fuel_type']].head(10)

## 5. Weather

ConPyro needs weather for each plot, so the next step attaches it to the plot table. FuelAnalyst supports two parts:

1. **Extraction** (`extract_nrcan_wx.extract_weather_data()`) reads gridded daily fire weather rasters and writes a
   time series for each plot location. It needs the gridded weather archive on your computer, so this tutorial does
   not run it.
2. **Merging** (`merge_wx_data.merge_weather_data()`) joins that time series to the plot table.

### Extraction (not run here)

The call below shows how extraction is set up. Variables are defined first so you can see what each parameter does.
Replace the archive path with the location of your own gridded data.

```python
from fuel_analyst.tools import extract_nrcan_wx

extract_nrcan_wx.extract_weather_data(
    wx_data_dir=r'path/to/gridded_fire_weather_archive',  # folder of gridded daily weather rasters
    out_dir=OUT_DIR / 'wx_extract',
    out_file_name='wx_data',
    plot_pts_path=FA_DIR / 'conpyro_df.csv',              # table holding each plot's coordinates
    plot_id_col_name='unique_id',
    lat_col_name='lat',                                   # decimal degrees, WGS84
    lon_col_name='long',
    wx_var_list=['temp', 'pcp', 'rh', 'ws', 'wd', 'ffmc', 'dmc', 'dc', 'bui', 'isi', 'fwi'],
    start_year=2005,
    end_year=2024,
    use_bcws_dates=False,                                 # True uses BCWS fire season dates for each plot's ecodivision
    start_month_day=501,                                  # MMDD; required when use_bcws_dates is False
    end_month_day=930,
    calculate_percentiles=True,
    percentile_var_list=['fwi'],
)
```

If your plot locations are wrong, the extracted weather will be wrong.

### The example weather

The example dataset includes real daily weather for the 20 plots (May to September, 2005 to 2024), already extracted.
The repository stores it already merged with plot attributes (and without FBP fuel types), so we keep only the
weather columns. That gives us what extraction would have produced for these plots.

In [ ]:
WX_DIR = OUT_DIR / 'wx_extract'
WX_DIR.mkdir(parents=True, exist_ok=True)

WX_COLS = ['unique_id', 'wx_date', 'temp', 'pcp', 'rh', 'ws', 'wd', 'ffmc', 'dmc', 'dc', 'bui', 'isi', 'fwi', 'fwi_prcnt']
wx_data = pd.read_csv(DATA_DIR / 'weather' / 'conpyro_df_wx.csv', usecols=WX_COLS)
wx_data.to_csv(WX_DIR / 'wx_data.csv', index=False)

print(f'{len(wx_data):,} daily records for {wx_data["unique_id"].nunique()} plots')
wx_data.head()

### Visualize the fire weather

To see what fire weather looks like at these plots, the next cell averages each weather variable by plot and year. Each
grey line is one plot. The coloured line is the median across plots, and the ribbon is the interquartile range.

In [ ]:
wx_data['year'] = wx_data['wx_date'].astype(str).str[:4].astype(int)
wx_by_year = (
    wx_data.groupby(['unique_id', 'year'])
    .agg(temp=('temp', 'mean'), pcp=('pcp', 'sum'), rh=('rh', 'mean'), fwi=('fwi', 'mean'))
    .reset_index()
)

VARIABLES = {
    'temp': ('Mean temperature', '\u00b0C', '#c0392b'),
    'pcp': ('Total precipitation', 'mm', '#2471a3'),
    'rh': ('Mean relative humidity', '%', '#148f77'),
    'fwi': ('Mean Fire Weather Index', 'index', '#b9770e'),
}
years = np.sort(wx_by_year['year'].unique())

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, (var, (label, unit, colour)) in zip(axes.flat, VARIABLES.items()):
    wide = wx_by_year.pivot(index='year', columns='unique_id', values=var)  # rows = years, columns = plots
    ax.plot(wide.index, wide.values, color='0.65', lw=0.8, alpha=0.6)        # one faint line per plot
    ax.fill_between(wide.index, wide.quantile(0.25, axis=1), wide.quantile(0.75, axis=1), color=colour, alpha=0.2, lw=0)
    ax.plot(wide.index, wide.median(axis=1), color=colour, lw=2.4)
    ax.set_title(label, fontsize=11, loc='left', pad=6)
    ax.set_ylabel(unit)
    ax.grid(alpha=0.25, lw=0.6)
    ax.margins(x=0.01)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel('Year')
axes[0, 0].set_xticks(years[::2])
fig.suptitle(f'Across-plot variability in annual weather, {years.min()}\u2013{years.max()}', x=0.02, ha='left')
fig.tight_layout()
plt.show()

Answer these from the figure, or compute them from `wx_by_year`.

1. Which years had the highest Fire Weather Index (FWI)?
2. What are the first and last years of weather data?

<details>
<summary><b>Answer: highest fire weather index years</b></summary>

Across plots, the highest median annual mean FWI was in **2022** (about 25.7), then **2017** (24.8) and **2021** (22.7). Check with `wx_by_year.groupby("year")["fwi"].median().sort_values(ascending=False)`.

</details>

<details>
<summary><b>Answer: years of weather data</b></summary>

The first year is **2005** and the last is **2024**.

</details>

### Merge weather with the plot table

`merge_weather_data()` joins the weather to the plot table on `unique_id`. It can also assign each date a season
(`add_season=True`), using `summer_start` and `fall_start` as `MMDD` integers, and refine each plot's FBP fuel type
by season (`refine_fbp_seasonally=True`). Here, summer starts on 15 June and fall on 15 September.

In [ ]:
WX_MERGED = WX_DIR / 'conpyro_df_wx.csv'

merge_wx_data.merge_weather_data(
    plot_data_path=str(FBP_DIR / 'conpyro_df_fbp.csv'),
    plot_merge_col_name='unique_id',
    wx_data_path=str(WX_DIR / 'wx_data.csv'),
    wx_merge_col_name='unique_id',
    wx_merge_vars=None,              # None merges every weather column
    out_path=str(WX_MERGED),
    add_season=True,
    refine_fbp_seasonally=True,
    summer_start=615,
    fall_start=915,
)

In [ ]:
conpyro_df_wx = pd.read_csv(WX_MERGED, low_memory=False)
print(f'{len(conpyro_df_wx):,} rows: one per plot per weather day')
conpyro_df_wx[['unique_id', 'wx_date', 'season', 'fbp_fuel_type', 'temp', 'rh', 'ws', 'ffmc', 'fwi']].head()

## 6. ConPyro and FOFEM

The last step runs `run_conpyro()`. For every plot and weather day it models:

- **fire behaviour**: crown fire initiation and spread with ConPyro, then flame length, height, tilt, residence time,
  and depth with flame-components;
- **tree mortality**: post-fire mortality with FOFEM, using the tree table; and
- **emissions**: fuel consumption and emissions with pyfofem, using the FOFEM consumption table.

The default settings (`default_fire_modelling_dict`) turn all of these on. Weather columns are required: `ws`, `wd`,
`ffmc`, `dmc`, `bui`, `temp`, and `rh` for ConPyro, and `ws` and `temp` for FOFEM mortality.

### Choose the weather days to model

The merged table has about 61,000 plot-days. To keep this tutorial quick, we model only each plot's most extreme fire
weather: days at or above the 97th FWI percentile (`fwi_prcnt`). For your own analysis, choose the days that answer
your question, or run them all.

In [ ]:
extreme_days = conpyro_df_wx[conpyro_df_wx['fwi_prcnt'] >= 97]
extreme_path = WX_DIR / 'conpyro_df_wx_extreme.csv'
extreme_days.to_csv(extreme_path, index=False)
print(f'Modelling {len(extreme_days):,} of {len(conpyro_df_wx):,} plot-days')

In [ ]:
CONPYRO_DIR = OUT_DIR / 'conpyro'

fire_modelling_params = copy.deepcopy(default_fire_modelling_dict)  # start from the defaults and change what you need

run_conpyro(
    out_dir=CONPYRO_DIR,
    in_conpyro_csv_path=extreme_path,
    in_tree_csv_path=FA_DIR / 'master_tree_df.csv',
    in_fofem_consume_csv_path=FA_DIR / fofem_units.consume_filename('SI'),  # must match the emissions unit system (SI)
    fire_modelling_params=fire_modelling_params,
)

### The results

`run_conpyro()` writes one folder per model plus a combined table. Like FuelAnalyst, it also writes an error report
(`run_conpyro_errors.csv`); an empty file means no problems.

In [ ]:
for path in sorted(CONPYRO_DIR.rglob('*.csv')):
    print(path.relative_to(CONPYRO_DIR))

In [ ]:
results = pd.read_csv(CONPYRO_DIR / 'conpyro_mortality_emissions_results.csv', low_memory=False)
print(f'{len(results):,} plot-days from {results["unique_id"].nunique()} plots, {results.shape[1]} columns')
results[['unique_id', 'wx_date', 'fwi', 'fire_type', 'hfi', 'flame_length_m', 'prop_dead_ba_crnsch', 'CO2F_gpm2']].head()

`fire_type` is the fire type ConPyro predicts (1 = surface, 2 = passive crown, 3 = active crown), `hfi` is head fire
intensity (kW/m), `flame_length_m` is flame length, `prop_dead_ba_crnsch` is the proportion of basal area killed
(FOFEM crown scorch mortality), and `CO2F_gpm2` is flaming-phase CO2 emitted (g/m2, SI units). The table has many more
columns; list them with `results.columns`.

### Plots that were skipped

We sent 1,841 plot-days to `run_conpyro()`, but fewer came back. `run_conpyro()` drops plots whose stand inputs are
empty, because there is no live overstory for ConPyro to model. Those plots do not appear in the results and are not
listed in `run_conpyro_errors.csv`, so it is worth checking for them yourself:

In [ ]:
skipped = sorted(set(extreme_days['unique_id']) - set(results['unique_id']))
print(f'{len(skipped)} plots were not modelled:')
conpyro_df.loc[conpyro_df['unique_id'].isin(skipped), ['unique_id', 'stand_density', 'stand_type', 'pct_cnfr', 'ht_ovrstry_live']]

<details>
<summary><b>Answer: why were these plots skipped?</b></summary>

All three are burned plots with **no live overstory trees** after the fire: `stand_type` and `ht_ovrstry_live` are empty and `pct_cnfr` is 0. With no live trees there is no canopy fuel for ConPyro to model, so `run_conpyro()` drops the plot (it removes rows with an empty `stand_density` or `stand_type`).

</details>

### Compare treated and untreated plots

As an example, compare the treated and untreated plots on these extreme days. The burned plots are left out because
their tree data describe the stand *after* the fire.

In [ ]:
FIRE_TYPES = {1: 'surface', 2: 'passive crown', 3: 'active crown'}

treatment = results['unique_id'].str.extract(r'_(burned|treated|untreated)_')[0].rename('treatment')
unburned = results[treatment.isin(['treated', 'untreated'])]
unburned_treatment = treatment[unburned.index]

summary = unburned.groupby(unburned_treatment).agg(
    plot_days=('unique_id', 'size'),
    median_hfi_kWpm=('hfi', 'median'),
    median_flame_length_m=('flame_length_m', 'median'),
    median_basal_area_killed=('prop_dead_ba_crnsch', 'median'),
    median_flaming_CO2_gpm2=('CO2F_gpm2', 'median'),
)
display(summary.round(2))

fire_type_share = pd.crosstab(unburned_treatment, unburned['fire_type'].map(FIRE_TYPES), normalize='index')
display(fire_type_share.round(2))

## Where to go next

- Change `default_fire_modelling_dict` to choose other crown fire initiation or spread models. `cfo_model=10` selects
  CFIS, and `examples/run_fuel_analyst_cfis_demo.py` shows that run.
- Try your own data: export your Survey123 forms into the five-folder structure and start again at section 3. Read
  `fuelanalyst_errors.csv` before anything else.
- See `docs/CODEBASE.md` for how the pipeline is organized, and `docs/survey123/` for the field forms and sampling
  protocols behind the input data.